# Space Launch Data Acquisition via API Pagination

**Notebook 1** - Data Collection Phase

## Purpose
This notebook collects comprehensive historical rocket launch data from the Space Devs Launch Library 2 API using automated pagination to retrieve all available records.

## Authors
- **Phillip Roman** - Pagination implementation, rate limiting, and automated compression
- **Jillian Kunze** - Initial API exploration and date filtering prototype

## Workflow Position
1. **This Notebook** → Collect raw launch data via API
   * *Full Collection:* ~5-6 hours
   * *Test Mode:* ~8 minutes (12 calls)
2. Data Cleaning → Extract parameters from raw data:
   - 2a. Extract rocket parameters
   - 2b. Extract launch parameters
   - 2c. Extract mission parameters
3. Data Merging → Merge all cleaned datasets

## Key Features
- **Automated Pagination:** Iterates through API results to capture the complete dataset.
- **Rate Limiting Compliance:** Built-in pauses (15 requests/hour) to respect API constraints.
- **Safe Test Mode:** Validates logic on a small sample (12 calls) and saves to a distinct `_TEST` file to prevent overwriting production data.
- **Automatic Compression:** Saves output as `.json.zip` to solve GitHub file size limits.
- **Robust Directory Handling:** Automatically creates necessary folders, ensuring compatibility with Google Colab and local environments.

## Output
- `raw_baseline_launches_Group7.json.zip` - **Full Collection:** Compressed dataset containing all historical launches.
- `raw_baseline_launches_Group7_TEST.json.zip` - **Test Mode:** Compressed sample dataset used for validation.

## Import Required Libraries

Load all necessary Python packages for API communication, data handling, and file operations.

In [22]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import dateutil.parser as dateparser
from pprint import pprint
import requests
import time
import json
import os
import zipfile

## Set Up API Query Parameters

Configure the API request settings and build the initial URL for data collection.

**Parameters:**
- **Mode:** detailed - Returns all nested objects for each launch
- **Limit:** 100 records per page (maximum allowed by API)
- **Ordering:** net (launch date) - Ascending chronological order

In [23]:
# API parameters
mode = 'mode=detailed'  # returns all related objects
limit = 'limit=100'  # this is the max!
ordering = 'ordering=net'  # ascending date order

# Assemble the full URL
current_url = "https://ll.thespacedevs.com/2.3.0/launches/previous/" + "?" + "&".join(
    (mode, limit, ordering)
)

print(f'Query URL: {current_url}')  # Visual check

Query URL: https://ll.thespacedevs.com/2.3.0/launches/previous/?mode=detailed&limit=100&ordering=net


## Configuration: Test Mode vs Full Collection

Set TEST_MODE to control the data collection scope.

**TEST_MODE = True:**
- Collects 12 API calls (1,000 launches)
- Pauses after 4 calls for 2 minutes
- Use for testing and validation

**TEST_MODE = False:**
- Collects ALL historical launches (7,000+ records)
- Pauses after 14 calls for 1hr 30 sec.
- Full collection takes approximately 5-6 hours

**Important:** Run this cell before the pagination loop to set your collection mode.

In [24]:

TEST_MODE = True  # Set to True for test run, False for full collection

# Set parameters
if TEST_MODE:
    max_calls = 12
    pause_after_call_num = 4
    pause_duration_in_seconds = 120  # 2 min
else:
    max_calls = None  # collects everything
    pause_after_call_num = 14  # API limit is 15 calls/hour
    pause_duration_in_seconds = 3630  # 1 hour 30 seconds

print(f"Running in {'TEST' if TEST_MODE else 'FULL COLLECTION'} mode")

Running in TEST mode


## Data Collection via Automated Pagination

This cell executes the main data collection workflow, including:

1. **API Pagination:** Automatically follows 'next' URLs until all launches are collected
2. **Rate Limiting:** Pauses every 14 calls (1hr 30 sec) to comply with API limits
3. **Network Timeout:** 60-second timeout to handle connection issues
4. **Progress Tracking:** Displays request count and cumulative totals
5. **Metadata Collection:** Records collector name, timestamp, and total count
6. **Automatic Compression & Saving:** Saves dataset to a compressed ZIP file to ensure GitHub compatibility

### Before Running:
No code changes are required. The script automatically handles directory creation for both local and cloud (Colab) environments.

**Optional:** Change `collector_name` variable if you wish to tag the file with a different user identifier.

### Expected Runtime:
- **Test Mode:** ~8 minutes (12 API calls)
- **Full Collection:** ~5-6 hours (due to API rate limiting)

### Output File:
- **Full:** `raw_baseline_launches_Group7.json.zip`
- **Test:** `raw_baseline_launches_Group7_TEST.json.zip`

---

**Pagination Logic Credit:**
Based on [Stack Overflow example](https://stackoverflow.com/questions/56206038/how-to-loop-through-paginated-api-using-python)

In [25]:
all_launches = []

api_call_count = 0

print("STARTING DATA ACQUISITION")
while current_url and (max_calls is None or api_call_count < max_calls):

    print(f"Request #: {api_call_count + 1}")

    try:
        response = requests.get(current_url, timeout=60)  # 60 to prevent network drop

        if response.status_code == 200:

            raw_data = response.json()

            all_launches.extend(raw_data['results'])

            # get next page url (None if last page)
            current_url = raw_data['next']

            print(f"Success! So far collected {len(all_launches)} total launches.")

            api_call_count += 1

            # pause
            should_pause = (api_call_count % pause_after_call_num == 0 and
                  current_url is not None)

            if max_calls is not None:  # check to avoid test run bug
                should_pause = should_pause and (api_call_count < max_calls)

            if should_pause:

                philly_tz = ZoneInfo('America/New_York')
                time_in_philly = datetime.now(philly_tz)
                length_of_pause = timedelta(seconds=pause_duration_in_seconds)
                resume_time = time_in_philly + length_of_pause

                print(f"{api_call_count} CALLS MADE. PAUSING FOR {pause_duration_in_seconds / 60:.0f} MIN.")
                print(f"Resuming at {resume_time.strftime('%I:%M:%S %p %Z')}")

                time.sleep(pause_duration_in_seconds)

                print("RESUMING API CALLS")

        else:
            print(f"Error! Status code: {response.status_code}")
            break

    except requests.exceptions.Timeout:
        print(f"Request timed out. Stopping collection.")
        break

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}. Stopping collection.")
        break

print(f"FINISHED: Collected {len(all_launches)} total launches!!!")

# update collector's name before running
collector_name = 'Group7'

if TEST_MODE:
    json_filename = f'raw_baseline_launches_{collector_name}_TEST.json'
else:
    json_filename = f'raw_baseline_launches_{collector_name}.json'

final_data = {
    'collector': collector_name,
    'total_launches': len(all_launches),
    'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'launches': all_launches
}

# option 1: Save to current notebook directory (for quick local testing)
# output_path = f"{json_filename}.zip"

# option 2: Save to Google Drive (For Colab users)
# output_path = f"/content/drive/MyDrive/DSCI511/Term Project/{json_filename}.zip"

# option 3 (Active): Save to project 'raw data' folder (Standard Repo Path)
zip_filename = f"{json_filename}.zip"
output_path = os.path.join("..", "data", "raw data", zip_filename)

print(f"Saving {len(all_launches)} launches to ZIP file...")

folder_path = os.path.dirname(output_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Created missing directory structure: {folder_path}")

try:
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr(json_filename, json.dumps(final_data, indent=4))

    print(f"File saved to: {output_path}")
    print("Save complete!")

except Exception as e:
    print(f"Error saving file: {e}")

STARTING DATA ACQUISITION
Request #: 1
Error! Status code: 429
FINISHED: Collected 0 total launches!!!
Saving 0 launches to ZIP file...
File saved to: ../data/raw data/raw_baseline_launches_Group7_TEST.json.zip
Save complete!


## Post-Acquisition Validation (Optional)

The following cells are for exploring the collected data and validating the API collection worked correctly.

**These are NOT required for the data acquisition workflow.**

In [26]:
print(f"Collected a total of {len(all_launches)} launches")

print("First launch:")
pprint(all_launches[:1])

Collected a total of 0 launches
First launch:
[]



### View the dates of first 10 launches

In [27]:
# view dates
for item in all_launches[:10]:
    print(item["net"])

### API Rate Limit Verification
Checks the API's current throttle status to confirm compliance with API rate limit and see remaining calls available in the current window.

Based on [docs](https://ll.thespacedevs.com/2.3.0/api-throttle/)

In [28]:
API_throttle_URL = "https://ll.thespacedevs.com/2.3.0/api-throttle/"

response_throttle = requests.get(API_throttle_URL)

throttle_data = response_throttle.json()
pprint(throttle_data)

{'current_use': 15,
 'ident': '34.55.160.19',
 'limit_frequency_secs': 3600,
 'next_use_secs': 2993,
 'your_request_limit': 15}
